In [ ]:
!pip install ultralytics -q
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.2 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
!mkdir -p /content/dataset/images
!mkdir -p /content/dataset/labels

!unzip -q "/content/drive/MyDrive/Dataset/imagestrain.zip" -d "/content/dataset/images"

print("✅ Images unzipped successfully!")

✅ Images unzipped successfully!


In [ ]:
import os
import cv2
from ultralytics import YOLO

# Colab Paths
IMAGE_DIR = '/content/dataset/images'
LABEL_DIR = '/content/dataset/labels'

# Load YOLO model
model = YOLO('yolov8n.pt')

# Your fixed ROI
ROI_STR = "0.2484 0.7399 0.4813 0.5047"
roi_xc, roi_yc, roi_w, roi_h = map(float, ROI_STR.split())

roi_x1 = roi_xc - (roi_w / 2)
roi_x2 = roi_xc + (roi_w / 2)
roi_y1 = roi_yc - (roi_h / 2)
roi_y2 = roi_yc + (roi_h / 2)

USE_STANDARD_YOLO_EMPTY_FILES = True
vehicle_classes = [2, 3, 5, 7]

cars_found = 0
empty_roads = 0

print("Starting Auto-Annotation on Colab GPU...")

# To handle unzipped folders safely (sometimes zip puts files inside a subfolder)
image_files = []
for root, dirs, files in os.walk(IMAGE_DIR):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_files.append(os.path.join(root, file))

for img_path in image_files:
    img_file = os.path.basename(img_path)
    txt_filename = os.path.splitext(img_file)[0] + '.txt'
    txt_path = os.path.join(LABEL_DIR, txt_filename)

    results = model(img_path, verbose=False)
    vehicle_in_roi = False

    for box in results[0].boxes:
        if int(box.cls[0]) in vehicle_classes:
            vx_center, vy_center, _, _ = box.xywhn[0].tolist()

            if (roi_x1 <= vx_center <= roi_x2) and (roi_y1 <= vy_center <= roi_y2):
                vehicle_in_roi = True
                break

    with open(txt_path, 'w') as f:
        if vehicle_in_roi:
            f.write(f"1 {ROI_STR}\n")
            cars_found += 1
        else:
            if USE_STANDARD_YOLO_EMPTY_FILES:
                pass
            else:
                f.write(f"0 {ROI_STR}\n")
            empty_roads += 1

print("\n✅ Labeling Complete!")
print(f"Total Processed: {cars_found + empty_roads}")
print(f"Vehicles in ROI: {cars_found}")
print(f"Empty ROIs: {empty_roads}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Starting Auto-Annotation on Colab GPU...

✅ Labeling Complete!
Total Processed: 12128
Vehicles in ROI: 1881
Empty ROIs: 10247


In [ ]:
import shutil
from google.colab import files

print("Zipping labels...")
shutil.make_archive("/content/generated_labels", 'zip', "/content/dataset/labels")

print("Downloading to your computer...")
files.download("/content/generated_labels.zip")

Zipping labels...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>